In [17]:
# --- STEP 1: Installation & Imports ---
!pip install pandasql

import pandas as pd
import numpy as np
from pandasql import sqldf

# --- STEP 2: Loading Raw Data ---
jobs_df = pd.read_csv('ai_job_replacement_2020_2026_v2.csv')
costs_df = pd.read_csv('cost_of_living.csv')
market_df = pd.read_csv('ai_job_market_2026.csv')

print("Step 2: Datasets loaded. 📥")

# --- STEP 3: Cleaning & Mapping ---
country_map = {
    'united states': 'usa',
    'united kingdom': 'uk',
    'united arab emirates': 'uae',
    'south korea': 'korea, south'
}

jobs_df['country'] = jobs_df['country'].str.strip().str.lower().replace(country_map)
costs_df['country'] = costs_df['country'].str.strip().str.lower().replace(country_map)
jobs_df['job_role'] = jobs_df['job_role'].str.strip().str.lower()
market_df['Job_Title'] = market_df['Job_Title'].str.strip().str.lower()

# --- STEP 4: Smart Merging & "Hard" Duplicate Cleaning ---
# 1. Merge the first two
inter_df = pd.merge(jobs_df, costs_df, on=['country', 'year'], how='inner')

# 2. Merge with the third
master_df = pd.merge(inter_df, market_df, left_on='job_role', right_on='Job_Title', how='left')

# --- THE MEGA FIX FOR SQL ERROR ---
# Move everything to lowercase first to find duplicates easier
master_df.columns = [col.lower() for col in master_df.columns]

# Now we drop any columns that are literally duplicated in name
master_df = master_df.loc[:, ~master_df.columns.duplicated()]

# Manually drop the specific troublemakers if they still exist
cols_to_remove = ['job_title', 'industry_y', 'ai_adoption_level_y'] # _y comes from merge duplicates
master_df.drop(columns=[c for c in cols_to_remove if c in master_df.columns], inplace=True)

print(f"Step 4: Master Table cleaned from ALL duplicates. Rows: {len(master_df)} ✅")

# --- STEP 5: Feature Engineering ---
master_df['prosperity_score'] = (master_df['salary_after_usd'] / master_df['cost_of_living_index']) * 100

# --- STEP 6: SQL Analysis ---
pysqldf = lambda q: sqldf(q, globals())

# Note: All column names are now LOWERCASE for SQL
final_query = """
SELECT
    country,
    job_role,
    COUNT(*) as total_jobs,
    ROUND(AVG(salary_after_usd), 2) as avg_salary,
    ROUND(AVG(prosperity_score), 2) as avg_prosperity
FROM master_df
GROUP BY country, job_role
HAVING total_jobs > 5
ORDER BY avg_prosperity DESC
LIMIT 10;
"""

# Executing SQL
analysis_results = pysqldf(final_query)

# --- STEP 7: Export ---
master_df.to_csv('final_ai_analytics_2026.csv', index=False)

print("\n--- 🏆 FINAL RESULTS (No Duplicates Anymore!) ---")
print(analysis_results)

Step 2: Datasets loaded. 📥
Step 4: Master Table cleaned from ALL duplicates. Rows: 15000 ✅

--- 🏆 FINAL RESULTS (No Duplicates Anymore!) ---
  country              job_role  total_jobs  avg_salary  avg_prosperity
0   india     financial analyst         150    92404.55       422684.59
1   india          truck driver         153    94740.71       421843.09
2   india               teacher         183    91582.71       413952.43
3   india  customer support rep         177    92331.64       411309.74
4   india  marketing specialist         169    88335.64       397592.49
5   india            accountant         165    87754.63       396201.17
6   india     software engineer         157    88179.35       394765.46
7   india          data analyst         154    87988.69       392279.79
8   india            hr manager         179    86219.45       391282.22
9   india   mechanical engineer         153    86607.84       390098.04
